# Chapter 12 — Weekly Iterative Fine-Tuning

**Goal:** Fine-tune GRU4Rec V9 on incremental Feb 2020 data in 4 weekly steps.

Each weekly run:
1. Computes item-popularity drift (JSD) vs. Jan 2020 baseline
2. Fine-tunes 5 epochs from the current checkpoint at `lr=1e-4`
3. Evaluates NDCG@20 on validation set
4. Promotes checkpoint only if NDCG@20 improves ≥ 0.0005

**Runtime:** ~8h total on T4 GPU. State saved to Drive after each week — edit `WEEKS_TO_RUN` to resume if the session drops.

## Prerequisites
- Google Drive mounted with REES46 CSVs at `MyDrive/rees46/`
- Val sessions parquet at `MyDrive/recosys_sessions/val_sessions.parquet`
- Colab runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ── Cell 1: Mount Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Clone / update repo + install deps ───────────────────────────────
import os

if not os.path.exists('/content/RecoSys'):
    !git clone -b dev https://github.com/sbnikhil/RecoSys.git /content/RecoSys
else:
    !git -C /content/RecoSys pull origin dev

%cd /content/RecoSys
!pip install torch torchvision pandas pyarrow numpy huggingface_hub faiss-cpu -q

In [ ]:
# ── Cell 3: Verify GPU ───────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — switch to T4 GPU runtime!')

In [ ]:
# ── Cell 4: Download model artifacts from HF Hub ────────────────────────────
from huggingface_hub import hf_hub_download
import shutil, os

os.makedirs('model', exist_ok=True)

for fname in ['model_inference.pt', 'vocabs.pkl']:
    local = f'model/{fname}'
    if os.path.exists(local):
        print(f'  Already exists: {fname}')
        continue
    tmp = hf_hub_download(
        repo_id='manojarulmurugan/recosys-models',
        filename=fname,
        repo_type='dataset',
    )
    shutil.copy(tmp, local)
    print(f'  Downloaded {fname}: {round(os.path.getsize(local)/1e6, 1)} MB')

In [ ]:
# ── Cell 5: Copy CSVs to local SSD (faster I/O than FUSE mount) ─────────────
import shutil, os, time

src = '/content/drive/MyDrive/rees46'
dst = '/content/rees46_local'
os.makedirs(dst, exist_ok=True)

for f in ['2020-Jan.csv.gz', '2020-Feb.csv.gz']:
    d = f'{dst}/{f}'
    if os.path.exists(d):
        print(f'  Already local: {f}')
        continue
    s = f'{src}/{f}'
    if not os.path.exists(s):
        print(f'  NOT FOUND: {s}')
        continue
    sz = os.path.getsize(s) / 1e6
    print(f'  Copying {f} ({sz:.0f} MB)...', end=' ', flush=True)
    t = time.time()
    shutil.copy2(s, d)
    print(f'done in {time.time()-t:.0f}s')

print('\nLocal files:', sorted(os.listdir(dst)))

In [ ]:
# ── Cell 6+7: Run all weeks with streaming output ───────────────────────────
# The script handles the week loop internally (--weeks accepts multiple values).
# Edit WEEKS_TO_RUN to resume from week 3 or 4 if the session timed out.
import subprocess, sys, time

WEEKS_TO_RUN = [1, 2, 3, 4]  # ← change to e.g. [3, 4] to resume

start = time.time()

proc = subprocess.Popen(
    [
        sys.executable,
        'scripts/retrain/run_weekly_pipeline.py',
        '--feb-csv',      '/content/rees46_local/2020-Feb.csv.gz',
        '--jan-csv',      '/content/rees46_local/2020-Jan.csv.gz',
        '--base-ckpt',    'model/model_inference.pt',
        '--vocabs-path',  'model/vocabs.pkl',
        '--val-sessions', '/content/drive/MyDrive/recosys_sessions/val_sessions.parquet',
        '--ckpt-dir',     '/content/drive/MyDrive/recosys_weekly',
        '--weeks',        *[str(w) for w in WEEKS_TO_RUN],
    ],
    cwd='/content/RecoSys',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
elapsed = (time.time() - start) / 60
print(f'\nTotal time: {elapsed:.1f} min  |  Exit code: {proc.returncode}')
if proc.returncode != 0:
    print('Pipeline failed — scroll up for the error.')

In [ ]:
# ── Cell 8: Inspect pipeline state ──────────────────────────────────────────
import json, os

state_path = '/content/drive/MyDrive/recosys_weekly/pipeline_state.json'
if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    print('Pipeline state:')
    print(json.dumps(state, indent=2))
else:
    print('No pipeline_state.json yet — run Cell 6+7 first')

In [ ]:
# ── Cell 9 (optional): Upload best checkpoint to HF Hub ────────────────────
import subprocess, sys

HF_TOKEN = ''  # ← paste your HF token

if HF_TOKEN:
    subprocess.run([
        sys.executable, 'scripts/deploy/upload_to_hf_hub.py',
        '--token',        HF_TOKEN,
        '--hf-username',  'manojarulmurugan',
        '--repo-name',    'recosys-models',
        '--model-dir',    '/content/drive/MyDrive/recosys_weekly',
    ], cwd='/content/RecoSys')
else:
    print('Set HF_TOKEN above to push the checkpoint to HF Hub')